In [ ]:
import os
import json
import time
import random
import logging
from datasets import load_dataset, Dataset, DatasetDict
from translatepy import Translator
from translatepy.translators import YandexTranslate
from tqdm.auto import tqdm

# Создаём переводчик
yandex = YandexTranslate()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [2]:
SOURCE_REPO_ID = "DeepPavlov/qrecc"
LOCAL_SAVE_PATH = "./qrecc_ru"
CACHE_FILE = "translation_cache_qrecc.jsonl"
SPLITS = ['train', 'test']

In [6]:
# ---------- Кэш переводов ----------
def load_cache():
    cache = {}
    if not os.path.exists(CACHE_FILE):
        return cache
    try:
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    except Exception as e:
        logging.error(f"Failed to load cache: {e}")
    return cache

def append_cache(text, translation):
    try:
        with open(CACHE_FILE, "a", encoding="utf-8") as f:
            f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")
    except Exception as e:
        logging.error(f"Failed to append cache: {e}")

translation_cache = load_cache()

def translate_text(text, retries=3, delay=3):
    """Перевод строки с кэшированием и тремя попытками."""
    if not isinstance(text, str) or text.strip() == "":
        return "", True
    if text in translation_cache:
        return translation_cache[text], True

    for attempt in range(retries):
        try:
            time.sleep(0.5 if attempt == 0 else delay)
            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)
            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True
        except Exception as e:
            logging.warning(f"Translation error: '{text[:50]}...'. Attempt {attempt+1}/{retries}. Error: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
    logging.error(f"Failed to translate: '{text[:50]}...'")
    return "", False


In [12]:
def translate_context(context_list):
    """
    Переводит список сообщений контекста.
    Возвращает список словарей вида:
    {"role": "user/assistant", "content_ru": "переведённый текст"}
    """
    if not context_list:
        return [], True

    translated = []
    all_success = True
    for msg in context_list:
        if isinstance(msg, dict):
            content = msg.get("content", "")
            role = msg.get("role", "user")
        else:
            content = str(msg)
            role = "user"

        content_ru, ok = translate_text(content)
        all_success = all_success and ok
        translated.append({
            "role": role,
            "content_ru": content_ru
        })
    return translated, all_success

def translate_example(example):
    success = True

    q_ru, ok = translate_text(example['question'])
    success = success and ok

    rw_ru, ok = translate_text(example['rewrite'])
    success = success and ok

    a_ru, ok = translate_text(example['answer'])
    success = success and ok

    ctx_ru, ok = translate_context(example['context'])
    success = success and ok

    return {
        'conversation_id': example.get('conversation_id', ''),
        'question': example['question'],
        'question_ru': q_ru,
        'rewrite': example['rewrite'],
        'rewrite_ru': rw_ru,
        'answer': example['answer'],
        'answer_ru': a_ru,
        'answer_url': example.get('answer_url', ''),
        'context': example['context'],  # оставляем исходный JSON как есть
        'context_ru': ctx_ru,
        '_success': success
    }

def process_split(split_name, source_split, progress_file):
    translated_records = []
    failed_indices = set()

    # Загрузка прогресса
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if rec.get('_failed', False):
                        failed_indices.add(rec['_index'])
                    else:
                        translated_records.append(rec)
                except:
                    continue
        logging.info(f"[{split_name}] Resuming. {len(translated_records)} ok, {len(failed_indices)} failed.")

    start_index = len(translated_records) + len(failed_indices)
    total = len(source_split)

    if start_index < total:
        logging.info(f"[{split_name}] Starting from index {start_index}...")
        with open(progress_file, "a", encoding="utf-8") as f:
            pbar = tqdm(
                enumerate(source_split.select(range(start_index, total))),
                desc=f"Translating {split_name}",
                total=total - start_index
            )
            for idx, example in pbar:
                global_idx = start_index + idx
                if global_idx in {r.get('_index', -1) for r in translated_records}:
                    continue

                translated = translate_example(example)
                record_out = {
                    '_index': global_idx,
                    '_failed': not translated['_success'],
                    **translated
                }
                del record_out['_success']

                f.write(json.dumps(record_out, ensure_ascii=False) + "\n")
                f.flush()

                if not record_out['_failed']:
                    translated_records.append(record_out)
                else:
                    failed_indices.add(global_idx)

    # Сбор успешных
    all_successful = []
    if os.path.exists(progress_file):
        with open(progress_file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    if not rec.get('_failed', False):
                        rec.pop('_index', None)
                        rec.pop('_failed', None)
                        all_successful.append(rec)
                except:
                    continue

    if not all_successful:
        logging.error(f"[{split_name}] No successful records.")
        return None

    ok_cnt = len(all_successful)
    fail_cnt = total - ok_cnt
    logging.info(f"[{split_name}] Completed: {ok_cnt}/{total} successful ({fail_cnt} failed)")
    if fail_cnt > 0:
        logging.info(f"[{split_name}] To retry failed, delete/modify {progress_file} and run again.")

    return Dataset.from_list(all_successful)

In [ ]:
# ---------- Запуск ----------
logging.info(f"Loading dataset '{SOURCE_REPO_ID}'...")
source = load_dataset(SOURCE_REPO_ID)

translated_splits = {}
for split in SPLITS:
    if split not in source:
        logging.warning(f"Split '{split}' not found, skipping...")
        continue
    progress_path = f"translated_qrecc_{split}.jsonl"
    ds_translated = process_split(split, source[split], progress_path)
    if ds_translated is None:
        logging.error(f"Failed to process split {split}")
        break
    translated_splits[split] = ds_translated

if len(translated_splits) == len(SPLITS):
    final_dataset = DatasetDict(translated_splits)
    logging.info(f"Saving dataset to '{LOCAL_SAVE_PATH}'...")
    final_dataset.save_to_disk(LOCAL_SAVE_PATH)
    print(f"\n✅ Dataset saved to {LOCAL_SAVE_PATH}")

    # Пример
    ex = final_dataset['train'][0]
    print("\n--- EXAMPLE (train) ---")
    print(f"Question: {ex['question']}")
    print(f"Question RU: {ex['question_ru']}")
    print(f"Answer: {ex['answer']}")
    print(f"Answer RU: {ex['answer_ru']}")
    print(f"Context first message RU: {ex['context_ru'][0]['content_ru']}")
else:
    logging.error("Not all splits were successfully translated.")


In [ ]:
from datasets import DatasetDict
from huggingface_hub import login

login(token="YOUR_HF_TOKEN")

# 2. Загружаем локально сохранённый датасет
LOCAL_PATH = "./qrecc_ru"
REPO_ID = "DeepPavlov/qrecc_ru"   # или "ваш_username/qrecc_ru"

dataset = DatasetDict.load_from_disk(LOCAL_PATH)

# 3. Заливаем каждый сплит как отдельный конфиг
for split_name, ds in dataset.items():
    ds.push_to_hub(
        REPO_ID,
        config_name=split_name,      # train, test
        private=False,
        commit_message=f"Upload {split_name} split with Russian translations"
    )
    print(f"Загружен {split_name}")

print(f"Готово: https://huggingface.co/datasets/{REPO_ID}")

In [ ]:
from datasets import DatasetDict
from huggingface_hub import login

login(token="YOUR_HF_TOKEN")

LOCAL_PATH = "./qrecc_ru"
REPO_ID = "DeepPavlov/qrecc_ru"

dataset = DatasetDict.load_from_disk(LOCAL_PATH)

# Удаляем лишнее поле во всех сплитах
for split in dataset:
    if 'conversation_id' in dataset[split].column_names:
        dataset[split] = dataset[split].remove_columns(['conversation_id'])

# Заливаем как отдельные конфигурации
for split_name, ds in dataset.items():
    ds.push_to_hub(
        REPO_ID,
        config_name=split_name,
        private=False,
        commit_message=f"Upload {split_name} split with Russian translations (clean)"
    )
    print(f"Загружен {split_name}")

print(f"Готово: https://huggingface.co/datasets/{REPO_ID}")

Допереводим то, что упало с ошибкой

In [54]:
# -------------------- Кэш --------------------
def load_cache():
    cache = {}
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["text"]] = obj["translation"]
                except:
                    continue
    return cache

def append_cache(text, translation):
    with open(CACHE_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"text": text, "translation": translation}, ensure_ascii=False) + "\n")

translation_cache = load_cache()

# -------------------- Улучшенный переводчик --------------------
def is_numeric_string(s: str) -> bool:
    """Возвращает True, если строка не содержит букв (ни латинских, ни кириллических)."""
    return not bool(re.search(r'[A-Za-zА-Яа-я]', s))

def translate_text_robust(text, retries=3, delay=3):
    if not isinstance(text, str) or text.strip() == "":
        return "", True

    # Любая строка из цифр и разрешённых разделителей – не переводим
    if is_numeric_string(text):
        return text, True

    if text in translation_cache:
        return translation_cache[text], True

    last_exception = None
    for attempt in range(retries):
        try:
            time.sleep(0.5 if attempt == 0 else delay)
            result = yandex.translate(text, "ru")
            translated = str(result.result) if hasattr(result, 'result') else str(result)
            translation_cache[text] = translated
            append_cache(text, translated)
            return translated, True
        except Exception as e:
            last_exception = e
            error_str = str(e).lower()
            if any(code in error_str for code in ['502', '503', '504']):
                wait = delay * (attempt + 1)
                logging.warning(f"Server error '{text[:30]}...' attempt {attempt+1}/{retries}. Wait {wait}s")
                time.sleep(wait)
            elif '429' in error_str:
                wait = delay * 4 + 10
                logging.warning(f"Rate limit. Wait {wait}s")
                time.sleep(wait)
            else:
                time.sleep(delay)

    # Если длинный текст упал с 5xx – переводим по предложениям
    if '502' in str(last_exception).lower() and len(text) > SPLIT_LEN_THRESHOLD:
        logging.info(f"Long text failed with 5xx, splitting into sentences: '{text[:30]}...'")
        sentences = re.split(r'(?<=[.!?])\s+', text)
        if len(sentences) > 1:
            translated_parts = []
            all_ok = True
            for sent in sentences:
                part, ok = translate_text_robust(sent, retries=retries, delay=delay)
                if not ok:
                    all_ok = False
                    break
                translated_parts.append(part)
            if all_ok:
                full_trans = ' '.join(translated_parts)
                translation_cache[text] = full_trans
                append_cache(text, full_trans)
                return full_trans, True

    # Если ничего не помогло – возвращаем пустую строку и флаг неуспеха
    logging.error(f"Failed to translate: '{text[:50]}...' Error: {last_exception}")
    return "", False

# -------------------- Перевод одного примера --------------------
def translate_context(context_list):
    if not context_list:
        return [], True
    translated = []
    all_success = True
    for msg in context_list:
        content = msg.get("content", "")
        role = msg.get("role", "user")
        content_ru, ok = translate_text_robust(content)
        all_success = all_success and ok
        translated.append({"role": role, "content_ru": content_ru})
    return translated, all_success

def translate_example(example):
    success = True
    q_ru, ok = translate_text_robust(example['question'])
    success = success and ok
    rw_ru, ok = translate_text_robust(example['rewrite'])
    success = success and ok
    a_ru, ok = translate_text_robust(example['answer'])
    success = success and ok
    ctx_ru, ok = translate_context(example['context'])
    success = success and ok

    return {
        'question': example['question'],
        'question_ru': q_ru,
        'rewrite': example['rewrite'],
        'rewrite_ru': rw_ru,
        'answer': example['answer'],
        'answer_ru': a_ru,
        'answer_url': example.get('answer_url', ''),
        'context': example['context'],
        'context_ru': ctx_ru,
        '_success': success
    }


In [55]:
# -------------------- Повторная обработка упавших --------------------
def retranslate_failed_to_new_file(split_name, old_progress_file, new_progress_file, source_split):
    records = []
    with open(old_progress_file, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                records.append(json.loads(line))
            except:
                continue

    updated_records = []
    failed_count = 0
    retried_success = 0
    with open(new_progress_file, 'w', encoding='utf-8') as out_f:
        for record in tqdm(records, desc=f"Retrying failed in {split_name}"):
            if not record.get('_failed', False):
                out_f.write(json.dumps(record, ensure_ascii=False) + '\n')
                updated_records.append(record)
                continue

            idx = record['_index']
            orig_example = source_split[int(idx)]
            translated = translate_example(orig_example)
            success = translated['_success']
            del translated['_success']

            new_record = {
                '_index': idx,
                '_failed': not success,
                **translated
            }
            out_f.write(json.dumps(new_record, ensure_ascii=False) + '\n')
            updated_records.append(new_record)

            failed_count += 1
            if success:
                retried_success += 1

    logging.info(f"[{split_name}] Retried {failed_count}, succeeded {retried_success}, still failed {failed_count - retried_success}")
    return [r for r in updated_records if not r.get('_failed', False)]

In [ ]:
# -------------------- Запуск --------------------
source = load_dataset(SOURCE_REPO_ID)

final_splits = {}
for split in SPLITS:
    old_file = f"translated_qrecc_{split}.jsonl"
    new_file = f"translated_qrecc_{split}_v2.jsonl"

    if not os.path.exists(old_file):
        logging.warning(f"Old progress file {old_file} not found, skipping {split}")
        continue

    successful = retranslate_failed_to_new_file(split, old_file, new_file, source[split])

    clean = []
    for rec in successful:
        rec.pop('_index', None)
        rec.pop('_failed', None)
        rec.pop('conversation_id', None)
        clean.append(rec)

    final_splits[split] = Dataset.from_list(clean)

# Сохраняем локально (НЕ на HF)
final_dataset = DatasetDict(final_splits)
final_dataset.save_to_disk(LOCAL_SAVE_PATH)
print(f"Исправленный датасет сохранён локально в {LOCAL_SAVE_PATH}")